In [1]:
from IPython.display import Image

import astroemperor as emp
import numpy as np
import multiprocessing
np.random.seed(1234)


sim = emp.Simulation()
sim.load_data('HIP8923')  # folder read from /datafiles/

sim.cores__ = multiprocessing.cpu_count()

I couldnt grab the terminal size. Trying with pandas...
Terminal size with pandas successful!
                                                                                
                   ~~ Simulation Successfully Initialized ~~                    
                                                                                



Reading data from HIP8923_1_cor07.vels
Reading data from HIP8923_1_cor14.vels
Reading data from HIP8923_1_cor14T.vels




                                                                                




In [2]:
ntemps, nwalkers, nsweeps, nsteps = 6, 128, 500, 1

sim.set_engine('reddemcee')
sim.engine_config['setup'] = [ntemps, nwalkers, nsweeps, nsteps]

# Custom adaptation options for reddemcee engine
sim.engine_config['tsw_history'] = True
sim.engine_config['smd_history'] = True
sim.engine_config['adapt_tau'] = 100
sim.engine_config['adapt_nu'] = 1.0
sim.engine_config['adapt_mode'] = 'SAR'

# Manual ladder
sim.engine_config['betas'] = list(np.geomspace(1, 0.001, ntemps))
# sim.engine_config['betas'][-1] = 0  # Optionally, for exact evidence estimation

sim.run_config['burnin'] = 0.5  # its in niter


if True:
    sim.starmass = 	1.0086627  # 0.9686578 1.0486716  # FLAME
    sim.starmass_err = 0.0400069
    sim.instrument_names_RV = ['Cor07', 'Cor14', 'Cor14T']

    # Model options
    sim.jitter_prargs = [0, 5]  # Normal(0, 10)
    sim.eccentricity_prargs = [0, 0.1]  # Normal(0, 0.1)
    sim.acceleration = 0  # no linear trend


    # Offsets and Jitters manual boundaries
    for i in range(1, 4):
        sim.add_condition([f'Offset {i}', 'limits', [-50, 50]])
        sim.add_condition([f'Jitter {i}', 'limits', [1e-4, 15]])

    # we constrain a bit the planetary parameters
    sim.add_condition(['Period 1', 'limits', [5100, 5200]])
    sim.add_condition(['Amplitude 1', 'limits', [45, 50]])
    sim.add_condition(['Eccentricity 1', 'limits', [0, 0.02]])

    sim.add_condition(['Phase 1', 'limits', [4.9, 5.0]])
    sim.add_condition(['Longitude 1', 'limits', [0.9, 0.94]])

    sim.add_condition(['Inclination 1', 'limits', [0.40, 0.45]])
    sim.add_condition(['Omega 1', 'limits', [1.70, 1.74]])


    # We also constrain astrometric offsets
    for offset in ['RA', 'DE', 'PLX', 'pm RA', 'pm DE']:
        sim.add_condition([f'Offset {offset}', 'limits', [-1, 1]])  # usually -10, 10

    # Finally, we set some sensible priors for Hipparcos and Gaia
    sim.add_condition(['Jitter Gaia', 'limits', [1, 5]])  # fractional jitter
    sim.add_condition(['Jitter Gaia', 'prior', 'Normal'])
    sim.add_condition(['Jitter Gaia', 'prargs', [1, 0.1]])

    sim.add_condition(['Jitter Hipp', 'limits', [0, 5]])  # additive jitter


if True:
    sim.add_condition(['Period 1', 'init_pos', [5150, 5170]])
    sim.add_condition(['Amplitude 1', 'init_pos', [48, 49.5]])
    sim.add_condition(['Eccentricity 1', 'init_pos', [0.005, 0.014]])

    sim.add_condition(['Phase 1', 'init_pos', [4.94, 4.96]])
    sim.add_condition(['Longitude 1', 'init_pos', [0.91, 0.93]])

    sim.add_condition(['Inclination 1', 'init_pos', [0.43, 0.44]])
    sim.add_condition(['Omega 1', 'init_pos', [1.71, 1.73]])

    sim.add_condition(['Offset 1', 'init_pos', [-30.3, -30.2]])
    sim.add_condition(['Offset 2', 'init_pos', [-38.2, -38.1]])
    sim.add_condition(['Offset 3', 'init_pos', [29.05, 29.10]])

    sim.add_condition(['Jitter 1', 'init_pos', [1.710, 1.720]])
    sim.add_condition(['Jitter 2', 'init_pos', [2.81, 2.83]])
    sim.add_condition(['Jitter 3', 'init_pos', [0.93, 0.95]])


    sim.add_condition(['Offset RA', 'init_pos', [0.600, 0.608]])
    sim.add_condition(['Offset DE', 'init_pos', [0.400, 0.416]])
    sim.add_condition(['Offset PLX', 'init_pos', [0.016, 0.018]])
    sim.add_condition(['Offset pm RA', 'init_pos', [0.21, 0.23]])
    sim.add_condition(['Offset pm DE', 'init_pos', [-0.240, -0.235]])

    sim.add_condition(['Jitter Gaia', 'init_pos', [1.13, 1.14]])  # 0.01
    sim.add_condition(['Jitter Hipparcos', 'init_pos', [1.01, 1.02]])  # 0.01

In [3]:
sim.plot_posteriors['temps'] = [0]
sim.plot_gaussian_mixtures['plot'] = False
sim.plot_trace['plot'] = False
sim.plot_histograms['plot'] = False

In [4]:
sim.autorun(1, 1)

                              Offset block added, OffsetBlock


                              Jitter block added, JitterBlock


                              Keplerian block added, AstrometryKeplerianBlock 1


                              AstrometryOffset block added, AstrometryOffsetBlock


                              AstrometryJitter block added, AstrometryJitterBlock



Condition applied: Parameter Period 1 attribute limits set to [5100, 5200]

Condition applied: Parameter Period 1 attribute init_pos set to [5150, 5170]

Condition applied: Parameter Amplitude 1 attribute limits set to [45, 50]

Condition applied: Parameter Amplitude 1 attribute init_pos set to [48, 49.5]

Condition applied: Parameter Phase 1 attribute limits set to [4.9, 5.0]

Condition applied: Parameter Phase 1 attribute init_pos set to [4.94, 4.96]

Condition applied: Parameter Eccentricity 1 attribute limits set to [0, 0.02]

Condition applied: Parameter Eccentricity 1 attribute init_pos set to [0.005, 0.014]

100%|██████████| 3000/3000 [00:07<00:00, 396.99it/s]


temp_script.py took 7.586 seconds

 FAILED 
 multiprocessing.set_start_method(fork, force=True)

Method set to  fork
Autocorrelation tolerance=50 fails. Setting to 0.
                         Calculating Gaussian Mixtures                          


100%|██████████| 22/22 [00:01<00:00, 17.28it/s]



                                 ~~ Best Fit ~~                                 


Parameter              Value(max)  Range(±σ)           Prior               Limits
-------------------  ------------  ------------------  ------------------  --------------
Period 1                 5186.21   [-41.366, 13.79]    ~𝓤 (5100, 5200)     [5100, 5200]
Amplitude 1                48.697  [-1.39, 1.27]       ~𝓤 (45, 50)         [45, 50]
Phase 1                     4.989  [-0.04, 0.011]      ~𝓤 (4.9, 5.0)       [4.9, 5]
Eccentricity 1              0.006  [-0.006, 0.006]     ~𝓝 (0.0, 0.1)       [0, 0.02]
Longitude 1                 0.913  [-0.013, 0.012]     ~𝓤 (0.9, 0.94)      [0.9, 0.94]
Inclination 1               0.449  [-0.009, 0.001]     ~I (0.4, 0.45)      [0.4, 0.45]
Omega 1                     1.73   [-0.002, 0.01]      ~𝓤 (1.7, 1.74)      [1.7, 1.74]
Semi-Major Axis 1           5.953  [-0.139, 0.081]                         [0, 1000]
Mass 1                      9.858  [-0.225, 0.172]      

...



                             Plotting Beta Density                              


100%|██████████| 1/1 [00:00<00:00, 17.54it/s]




                           Plotting Temperature Rates                           


100%|██████████| 5/5 [00:00<00:00, 33.66it/s]




                           Plotting Astrometric Model                           
I couldnt grab the terminal size. Trying with pandas...
Terminal size with pandas successful!




Time Table
Time RUN                   : 00:00:08
Time POSTPROCESS           : 00:00:01
Time CALCULATE GM          : 00:00:01
Time plot_posteriors       : 00:00:04
Time plot_histograms       : 00:00:00
Time plot_keplerian_model  : 00:00:00
Time plot_betas            : 00:00:00
Time plot_beta_density     : 00:00:00
Time plot_rates            : 00:00:00
Time plot_astrometric_model: 00:00:01

(Past BIC) - (Present BIC) > 5

 BIC condition met!!
inf - 586.796 > 5


                                                                                
                              ~~ End of the Run ~~                              
                                                                                




In [5]:
z_ti = np.round(sim.sampler.get_evidence_ti(discard=sim.reddemcee_discard)[:2], 3)
z_ss = np.round(sim.sampler.get_evidence_ss(discard=sim.reddemcee_discard), 3)
z_hy = np.round(sim.sampler.get_evidence_hybrid(discard=sim.reddemcee_discard), 3)

print(f'Thermodynamic Integration logZ = {z_ti[0]} ±{z_ti[1]}')
print(f'Stepping Stones logZ           = {z_ss[0]} ±{z_ss[1]}')
print(f'Hybrid Method logZ             = {z_hy[0]} ±{z_hy[1]}')

Thermodynamic Integration logZ = -288.757 ±17.964
Stepping Stones logZ           = -286.36 ±0.346
Hybrid Method logZ             = -298.174 ±18.043


Thermodynamic Integration logZ = -308.941 ±2.876

Stepping Stones logZ           = -308.473 ±0.497

Hybrid Method logZ             = -302.723 ±3.042